# 🎓 Phân Tích Cảm Xúc Phản Hồi Sinh Viên
## Hệ Thống Hỗ Trợ Cải Thiện Chất Lượng Giảng Dạy

| Thông tin | Chi tiết |
|-----------|----------|
| **Môn học** | Nhập môn Trí tuệ Nhân tạo |
| **Chủ đề** | Phân loại cảm xúc văn bản tiếng Việt (Sentiment Analysis) |
| **Dataset** | [uitnlp/vietnamese_students_feedback](https://huggingface.co/datasets/uitnlp/vietnamese_students_feedback) |
| **Phương pháp** | TF-IDF + Logistic Regression / Naive Bayes / SVM |

---
### 📌 Pipeline tổng quan

```
Phản hồi sinh viên
       ↓
Tiền xử lý văn bản  (lowercase, loại nhiễu, stopwords)
       ↓
TF-IDF Vectorization  (biểu diễn văn bản thành vector số)
       ↓
Huấn luyện & So sánh mô hình  (LR / Naive Bayes / SVM)
       ↓
Đánh giá: Accuracy, F1-Macro, Cross-Validation
       ↓
Phân loại: Positive / Negative / Neutral
       ↓
Phân tích lỗi + Biểu đồ thống kê
       ↓
Báo cáo & Gợi ý cải thiện chất lượng giảng dạy
```

---
## ⚙️ Bước 0: Cài Đặt Môi Trường

> Sử dụng `datasets==2.18.0` để tương thích với dataset script-based của UIT-NLP.  
> **Sau khi chạy cell cài thư viện → Runtime → Restart runtime → Run all.**

In [ ]:
# [0.1] Thiết lập HuggingFace Token
from google.colab import userdata
import os

try:
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
    print("✅ HF_TOKEN đã được thiết lập.")
except Exception:
    print("⚠️  Không tìm thấy HF_TOKEN.")
    print("   → Biểu tượng 🔑 bên trái → Add new secret → Name: HF_TOKEN")

In [ ]:
# [0.2] Cài thư viện
!pip install "datasets==2.18.0" scikit-learn matplotlib seaborn wordcloud -q
print("✅ Cài đặt hoàn tất!")

---
## 📥 Bước 1: Tải Dataset

**Dataset UIT-VSFC** (Vietnamese Students' Feedback Corpus):
- Nguồn: Đại học Công nghệ Thông tin – ĐHQG TP.HCM
- ~16,000 câu phản hồi sinh viên về các môn học
- 3 nhãn cảm xúc: **0 = Negative**, **1 = Neutral**, **2 = Positive**
- Đã được chia sẵn: Train / Validation / Test

In [ ]:
# [1.1] Tải dataset từ HuggingFace
from datasets import load_dataset
import pandas as pd
import numpy as np

print("⏳ Đang tải dataset uitnlp/vietnamese_students_feedback ...")
dataset = load_dataset(
    "uitnlp/vietnamese_students_feedback",
    trust_remote_code=True
)
print("✅ Tải thành công!")
print(dataset)

In [ ]:
# [1.2] Chuyển sang DataFrame
df_train = dataset['train'].to_pandas()
df_test  = dataset['test'].to_pandas()
df_val   = dataset['validation'].to_pandas() if 'validation' in dataset else None

if df_val is not None:
    print(f"📊 Kích thước: Train={len(df_train)} | Val={len(df_val)} | Test={len(df_test)}")
else:
    print(f"📊 Kích thước: Train={len(df_train)} | Test={len(df_test)}")

LABEL = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
print("\n🏷️  Phân phối nhãn (Train):")
print(df_train['sentiment'].value_counts().sort_index()
      .rename(index={0:'Negative(0)', 1:'Neutral(1)', 2:'Positive(2)'}))
print("\n🔍 5 dòng mẫu:")
display(df_train.head())

---
## 🧹 Bước 2: Tiền Xử Lý Văn Bản (Text Preprocessing)

Tiền xử lý chuẩn hóa dữ liệu, giảm nhiễu trước khi đưa vào mô hình.

| Bước | Mô tả | Ví dụ |
|------|-------|-------|
| 1 | Chuyển về chữ thường | `"Thầy DẠY Hay"` → `"thầy dạy hay"` |
| 2 | Loại bỏ URL | `"xem tại http://..."` → `"xem tại"` |
| 3 | Loại bỏ số | `"lớp 10a"` → `"lớp a"` |
| 4 | Loại bỏ ký tự đặc biệt | `"hay!!!"` → `"hay"` |
| 5 | Loại bỏ khoảng trắng thừa | `"hay  lắm"` → `"hay lắm"` |
| 6 | Loại stopwords | Bỏ từ phổ biến ít mang nghĩa |

> ⚠️ **Quan trọng:** Từ phủ định `không, chưa, chẳng` được **GIỮ LẠI** vì ảnh hưởng trực tiếp đến cảm xúc.  
> Ví dụ: *"dạy **không** hay"* ≠ *"dạy hay"* — xóa "không" sẽ đảo ngược nghĩa hoàn toàn.

In [ ]:
# [2.1] Định nghĩa stopwords và hàm tiền xử lý
import re

VIETNAMESE_STOPWORDS = set([
    'và', 'là', 'của', 'có', 'được', 'trong', 'cho', 'với',
    'này', 'đó', 'các', 'những', 'một', 'đã', 'tôi', 'bạn', 'họ',
    'mà', 'về', 'theo', 'từ', 'hay', 'khi', 'thì', 'vì', 'nên',
    'để', 'nhưng', 'còn', 'cũng', 'lại', 'đây', 'rất', 'hơn',
    'như', 'thế', 'nào', 'ai', 'gì', 'sẽ', 'đến', 'ra', 'đi',
    'lên', 'xuống', 'vào', 'đang', 'bị', 'phải', 'nếu',
    'mình', 'ta', 'chúng', 'em', 'anh', 'chị', 'ông', 'bà'
    # GIỮ LẠI: 'không', 'chưa', 'chẳng' — từ phủ định quan trọng
])

def preprocess_text(text):
    """Tiền xử lý văn bản tiếng Việt cho bài toán sentiment analysis."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)   # Loại URL
    text = re.sub(r'\d+', '', text)                  # Loại số
    text = re.sub(r'[^\w\s]', ' ', text)             # Loại ký tự đặc biệt
    text = re.sub(r'\s+', ' ', text).strip()         # Chuẩn hóa khoảng trắng
    tokens = [t for t in text.split()
              if t not in VIETNAMESE_STOPWORDS and len(t) > 1]
    return ' '.join(tokens)

In [ ]:
# [2.2] Áp dụng tiền xử lý
print("⏳ Đang tiền xử lý văn bản...")
df_train['processed'] = df_train['sentence'].apply(preprocess_text)
df_test['processed']  = df_test['sentence'].apply(preprocess_text)
if df_val is not None:
    df_val['processed'] = df_val['sentence'].apply(preprocess_text)

print(f"✅ Hoàn tất! Đã xử lý {len(df_train):,} câu train, {len(df_test):,} câu test.")
print("\n📝 Ví dụ trước/sau tiền xử lý:")
print("-" * 70)
for i in range(3):
    lname = LABEL.get(df_train['sentiment'].iloc[i], '?')
    print(f"  [{lname}]")
    print(f"  Gốc : {df_train['sentence'].iloc[i]}")
    print(f"  Sau : {df_train['processed'].iloc[i]}")
    print()

---
## 🔢 Bước 3: Vector Hóa Văn Bản với TF-IDF

Máy tính không hiểu chữ — cần chuyển văn bản thành **vector số**.

### Công thức TF-IDF:

$$\text{TF-IDF}(t, d) = \text{TF}(t,d) \times \text{IDF}(t)$$

- **TF** *(Term Frequency)*: Tần suất từ `t` trong văn bản `d` → Từ xuất hiện nhiều → trọng số cao
- **IDF** *(Inverse Document Frequency)*: $\log\dfrac{N}{df(t)}$ → Từ phổ biến khắp tập dữ liệu → trọng số thấp

**Ý nghĩa:** Từ TF-IDF cao = đặc trưng riêng của văn bản đó, mang nhiều thông tin phân loại.

### Tại sao dùng Bigram `(1,2)`?
- **Unigram** `"không"`, `"hay"` → có thể mơ hồ
- **Bigram** `"không hay"` → rõ nghĩa hơn, nắm được ngữ cảnh cạnh nhau

In [ ]:
# [3.1] TF-IDF Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=15000,  # Giữ 15k đặc trưng quan trọng nhất
    ngram_range=(1, 2),  # Unigram + Bigram
    sublinear_tf=True,   # log(1+TF) — giảm ảnh hưởng từ lặp quá nhiều
    min_df=2,            # Bỏ từ chỉ xuất hiện 1 lần (quá hiếm)
    max_df=0.95          # Bỏ từ xuất hiện trong >95% văn bản (quá phổ biến)
)

# QUAN TRỌNG: fit_transform trên train, chỉ transform trên test
# → Tránh data leakage (rò rỉ thông tin tập test vào quá trình huấn luyện)
X_train = tfidf.fit_transform(df_train['processed'])
X_test  = tfidf.transform(df_test['processed'])
if df_val is not None:
    X_val = tfidf.transform(df_val['processed'])

y_train = df_train['sentiment']
y_test  = df_test['sentiment']
if df_val is not None:
    y_val = df_val['sentiment']

print("✅ TF-IDF hoàn tất!")
print(f"   Số đặc trưng (features)   : {X_train.shape[1]:,}")
print(f"   Ma trận Train              : {X_train.shape[0]:,} × {X_train.shape[1]:,}")
print(f"   Ma trận Test               : {X_test.shape[0]:,} × {X_test.shape[1]:,}")
print(f"   Mật độ sparse (train)      : {X_train.nnz/(X_train.shape[0]*X_train.shape[1]):.4%}")

---
## 🤖 Bước 4: Huấn Luyện & So Sánh Các Mô Hình

So sánh **4 mô hình** phân loại phổ biến trong NLP:

| Mô hình | Ý tưởng chính | Ưu điểm |
|---------|---------------|---------|
| **Logistic Regression** | Học ranh giới tuyến tính qua hàm softmax, tối ưu bằng gradient descent | Nhanh, dễ giải thích hệ số, tốt với TF-IDF |
| **Multinomial Naive Bayes** | Áp dụng định lý Bayes: $P(y|x) \propto P(y)\prod P(x_i|y)$, giả sử độc lập | Cực nhanh, baseline tốt cho text |
| **Complement Naive Bayes** | Cải tiến MNB: học từ các lớp *bù* thay vì lớp chính | Tốt hơn MNB khi dữ liệu mất cân bằng |
| **Linear SVM** | Tìm siêu phẳng tối đa hóa margin: $\max \frac{2}{\|w\|}$ | Hiệu quả cao trong không gian chiều lớn |

### Metrics đánh giá:
- **Accuracy**: $\frac{TP+TN}{\text{Total}}$ — Tỷ lệ dự đoán đúng tổng thể
- **Precision**: $\frac{TP}{TP+FP}$ — Trong số dự đoán là X, bao nhiêu đúng thật
- **Recall**: $\frac{TP}{TP+FN}$ — Trong số thật là X, mô hình tìm được bao nhiêu
- **F1-Macro**: Trung bình F1 của từng lớp — phù hợp khi dữ liệu **không cân bằng**

In [ ]:
# [4.1] Huấn luyện và đánh giá 4 mô hình
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, f1_score, precision_score, recall_score)
import time

models = {
    "Logistic Regression" : LogisticRegression(
        C=1.0, max_iter=1000, solver='lbfgs',
        multi_class='multinomial', random_state=42),
    "Multinomial NB"      : MultinomialNB(alpha=0.1),
    "Complement NB"       : ComplementNB(alpha=0.1),
    "Linear SVM"          : LinearSVC(C=1.0, max_iter=2000, random_state=42)
}

results = {}
print("=" * 70)
print(f"{'Mo hinh':<22} {'Accuracy':>9} {'F1-Macro':>9} {'Precision':>10} {'Recall':>8} {'Time':>7}")
print("=" * 70)

for name, model in models.items():
    t0 = time.time()
    model.fit(X_train, y_train)
    t1 = time.time()

    y_pred   = model.predict(X_test)
    acc      = accuracy_score(y_test, y_pred)
    f1_mac   = f1_score(y_test, y_pred, average='macro')
    prec_mac = precision_score(y_test, y_pred, average='macro', zero_division=0)
    rec_mac  = recall_score(y_test, y_pred, average='macro', zero_division=0)

    results[name] = {
        'model': model, 'y_pred': y_pred,
        'accuracy': acc, 'f1_macro': f1_mac,
        'precision': prec_mac, 'recall': rec_mac,
        'time': t1 - t0
    }
    print(f"  {name:<20} {acc:>9.4f} {f1_mac:>9.4f} {prec_mac:>10.4f} {rec_mac:>8.4f} {t1-t0:>5.2f}s")

print("=" * 70)
best_name = max(results, key=lambda k: results[k]['f1_macro'])
print(f"\n🏆 Mo hinh tot nhat (theo F1-Macro): {best_name}")
print(f"   Accuracy  = {results[best_name]['accuracy']:.4f}")
print(f"   F1-Macro  = {results[best_name]['f1_macro']:.4f}")

In [ ]:
# [4.2] Classification Report chi tiết mô hình tốt nhất
y_pred_best = results[best_name]['y_pred']
label_names = ['Negative', 'Neutral', 'Positive']

print(f"📋 Classification Report — {best_name}")
print("=" * 60)
print(classification_report(y_test, y_pred_best, target_names=label_names))

---
## 🔁 Bước 4b: Đánh Giá Bằng Cross-Validation (Kiểm Chứng Độ Tin Cậy)

Kết quả trên tập test 1 lần có thể bị **may rủi** do cách chia dữ liệu.  
**Stratified K-Fold Cross-Validation** chia dữ liệu thành K phần, lần lượt dùng mỗi phần làm tập validation → kết quả **ổn định và đáng tin hơn**.

```
Dữ liệu train
├── Fold 1: [VAL] [train] [train] [train] [train]
├── Fold 2: [train] [VAL] [train] [train] [train]
├── Fold 3: [train] [train] [VAL] [train] [train]
├── Fold 4: [train] [train] [train] [VAL] [train]
└── Fold 5: [train] [train] [train] [train] [VAL]
```
Kết quả cuối = **trung bình** của 5 lần đánh giá → giảm phương sai.

In [ ]:
# [4b] 5-Fold Stratified Cross-Validation cho tất cả mô hình
from sklearn.model_selection import StratifiedKFold, cross_validate

print("⏳ Đang chạy 5-Fold Cross-Validation (có thể mất vài phút)...")
print("=" * 65)
print(f"{'Mo hinh':<22} {'CV F1-Macro':>12} {'Std (±)':>9} {'CV Accuracy':>12}")
print("=" * 65)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

for name, model in models.items():
    scores = cross_validate(
        model, X_train, y_train,
        cv=cv,
        scoring={'f1_macro': 'f1_macro', 'accuracy': 'accuracy'},
        n_jobs=-1
    )
    f1_mean  = scores['test_f1_macro'].mean()
    f1_std   = scores['test_f1_macro'].std()
    acc_mean = scores['test_accuracy'].mean()
    cv_results[name] = {'f1_mean': f1_mean, 'f1_std': f1_std, 'acc_mean': acc_mean}
    print(f"  {name:<20} {f1_mean:>12.4f} {f1_std:>9.4f} {acc_mean:>12.4f}")

print("=" * 65)
print("\n💡 Nhận xét Cross-Validation:")
best_cv = max(cv_results, key=lambda k: cv_results[k]['f1_mean'])
print(f"   Mô hình ổn định nhất: {best_cv}")
print(f"   CV F1-Macro = {cv_results[best_cv]['f1_mean']:.4f} ± {cv_results[best_cv]['f1_std']:.4f}")
print(f"   (Std nhỏ = kết quả ổn định, ít bị ảnh hưởng bởi cách chia dữ liệu)")

---
## 🧪 Bước 4c: Thực Nghiệm So Sánh Cấu Hình TF-IDF

Câu hỏi thực nghiệm: **Unigram hay Bigram hay Trigram cho kết quả tốt hơn?**

Đây là loại phân tích thực nghiệm (*ablation study*) thường thấy trong báo cáo khoa học.

In [ ]:
# [4c] So sánh ngram_range trên mô hình tốt nhất
print("⏳ Đang thực nghiệm so sánh n-gram...")

best_model_class = type(results[best_name]['model'])
best_model_params = results[best_name]['model'].get_params()

ngram_configs = {
    'Unigram (1,1)'  : (1, 1),
    'Bigram  (1,2)'  : (1, 2),
    'Trigram (1,3)'  : (1, 3),
}

print("=" * 55)
print(f"So sanh N-gram — Mo hinh: {best_name}")
print("=" * 55)
print(f"{'Cau hinh':<18} {'Accuracy':>10} {'F1-Macro':>10}")
print("-" * 55)

for config_name, ngram in ngram_configs.items():
    vec_tmp = TfidfVectorizer(
        max_features=15000, ngram_range=ngram,
        sublinear_tf=True, min_df=2, max_df=0.95
    )
    Xtr_tmp = vec_tmp.fit_transform(df_train['processed'])
    Xte_tmp = vec_tmp.transform(df_test['processed'])

    mdl_tmp = best_model_class(**best_model_params)
    mdl_tmp.fit(Xtr_tmp, y_train)
    yp_tmp  = mdl_tmp.predict(Xte_tmp)

    acc_tmp = accuracy_score(y_test, yp_tmp)
    f1_tmp  = f1_score(y_test, yp_tmp, average='macro')
    marker  = " ← (dang dung)" if ngram == (1,2) else ""
    print(f"  {config_name:<16} {acc_tmp:>10.4f} {f1_tmp:>10.4f}{marker}")

print("=" * 55)
print("\n💡 Nhận xét: Bigram thường cải thiện hơn Unigram vì nắm được")
print("   cụm từ như 'không hay', 'rất tốt', 'dễ hiểu'.")
print("   Trigram đôi khi không cải thiện thêm do thưa dữ liệu.")

---
## 📊 Bước 5: Trực Quan Hóa Kết Quả

9 biểu đồ tổng hợp đánh giá toàn diện hiệu quả các mô hình.

In [ ]:
# [5.1] Vẽ 9 biểu đồ tổng hợp
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

matplotlib.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120

COLORS = {0: '#ef4444', 1: '#6b7280', 2: '#22c55e'}
SHORT  = ['LR', 'MNB', 'CNB', 'SVM']
mnames = list(results.keys())

fig = plt.figure(figsize=(20, 15))
fig.suptitle('Phan Tich Cam Xuc Phan Hoi Sinh Vien\n'
             'Vietnamese Students Feedback — Sentiment Analysis',
             fontsize=16, fontweight='bold', y=0.98)

# 1. Phân phối nhãn Train
ax1 = fig.add_subplot(3, 3, 1)
counts = df_train['sentiment'].value_counts().sort_index()
clrs   = [COLORS[i] for i in counts.index]
bars   = ax1.bar([LABEL[i] for i in counts.index], counts.values,
                 color=clrs, edgecolor='white', linewidth=1.5)
ax1.set_title('Phan phoi nhan - Tap Train', fontweight='bold')
ax1.set_ylabel('So luong mau')
for bar, val in zip(bars, counts.values):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+10,
             str(val), ha='center', fontweight='bold', fontsize=10)

# 2. Pie chart
ax2 = fig.add_subplot(3, 3, 2)
ax2.pie(counts.values, labels=[LABEL[i] for i in counts.index],
        colors=clrs, autopct='%1.1f%%', startangle=90,
        textprops={'fontsize': 10})
ax2.set_title('Ti le phan bo nhan (Train)', fontweight='bold')

# 3. So sánh Accuracy
ax3 = fig.add_subplot(3, 3, 3)
accs    = [results[m]['accuracy'] for m in mnames]
bcolors = ['#f59e0b' if m == best_name else '#3b82f6' for m in mnames]
bars3   = ax3.bar(SHORT, accs, color=bcolors, edgecolor='white')
ax3.set_title('So sanh Accuracy', fontweight='bold')
ax3.set_ylabel('Accuracy')
ax3.set_ylim(max(0, min(accs)-0.05), 1.0)
for bar, acc in zip(bars3, accs):
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
             f'{acc:.3f}', ha='center', fontweight='bold', fontsize=10)
ax3.legend(handles=[
    plt.Rectangle((0,0),1,1, color='#f59e0b', label='Tot nhat'),
    plt.Rectangle((0,0),1,1, color='#3b82f6', label='Con lai')
])

# 4. Confusion Matrix
ax4 = fig.add_subplot(3, 3, 4)
cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax4,
            xticklabels=list(LABEL.values()),
            yticklabels=list(LABEL.values()), cbar=False)
ax4.set_title(f'Confusion Matrix\n({best_name})', fontweight='bold')
ax4.set_xlabel('Du doan (Predicted)')
ax4.set_ylabel('Thuc te (Actual)')

# 5. Độ dài văn bản
ax5 = fig.add_subplot(3, 3, 5)
df_train['text_len'] = df_train['sentence'].apply(lambda x: len(str(x).split()))
for label in sorted(df_train['sentiment'].unique()):
    ax5.hist(df_train[df_train['sentiment']==label]['text_len'],
             bins=25, alpha=0.65, color=COLORS[label],
             label=LABEL[label], edgecolor='white')
ax5.set_title('Do dai van ban theo nhan', fontweight='bold')
ax5.set_xlabel('So tu trong cau')
ax5.set_ylabel('So luong')
ax5.legend()

# 6. Top 15 từ đặc trưng Positive (LR)
ax6 = fig.add_subplot(3, 3, 6)
lr_model   = results['Logistic Regression']['model']
feat_names = np.array(tfidf.get_feature_names_out())
coef = lr_model.coef_[2] if lr_model.coef_.shape[0] > 1 else lr_model.coef_[0]
top_idx = np.argsort(coef)[-15:]
ax6.barh(range(15), coef[top_idx], color='#22c55e', edgecolor='white')
ax6.set_yticks(range(15))
ax6.set_yticklabels(feat_names[top_idx], fontsize=9)
ax6.set_title('Top 15 tu dac trung Positive\n(Logistic Regression)', fontweight='bold')
ax6.set_xlabel('He so (Coefficient)')

# 7. Thực tế vs Dự đoán
ax7 = fig.add_subplot(3, 3, 7)
all_labels = sorted(y_test.unique())
x = np.arange(len(all_labels)); w = 0.35
ax7.bar(x-w/2, [sum(y_test==l) for l in all_labels], w,
        label='Thuc te', color='#3b82f6', edgecolor='white')
ax7.bar(x+w/2, [sum(y_pred_best==l) for l in all_labels], w,
        label='Du doan', color='#f59e0b', edgecolor='white')
ax7.set_xticks(x)
ax7.set_xticklabels([LABEL[l] for l in all_labels])
ax7.set_title('Thuc te vs Du doan (Test)', fontweight='bold')
ax7.set_ylabel('So luong')
ax7.legend()

# 8. So sánh F1-Macro
ax8 = fig.add_subplot(3, 3, 8)
f1s  = [results[m]['f1_macro'] for m in mnames]
bc8  = ['#f59e0b' if m == best_name else '#8b5cf6' for m in mnames]
bars8 = ax8.bar(SHORT, f1s, color=bc8, edgecolor='white')
ax8.set_title('So sanh F1-Macro', fontweight='bold')
ax8.set_ylabel('F1-Macro')
ax8.set_ylim(max(0, min(f1s)-0.05), 1.0)
for bar, f in zip(bars8, f1s):
    ax8.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
             f'{f:.3f}', ha='center', fontweight='bold', fontsize=10)

# 9. F1 per class
ax9 = fig.add_subplot(3, 3, 9)
f1_per = f1_score(y_test, y_pred_best, average=None)
ax9.bar(list(LABEL.values()), f1_per,
        color=[COLORS[i] for i in sorted(LABEL)], edgecolor='white')
ax9.set_title(f'F1-score theo tung lop\n({best_name})', fontweight='bold')
ax9.set_ylabel('F1-score')
ax9.set_ylim(0, 1.15)
for i, f in enumerate(f1_per):
    ax9.text(i, f+0.02, f'{f:.3f}', ha='center', fontweight='bold', fontsize=11)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('bieu_do_ket_qua.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Da luu: bieu_do_ket_qua.png")

---
## 📊 Bước 5b: Biểu Đồ Cross-Validation

In [ ]:
# [5b.1] Biểu đồ so sánh CV F1-Macro với khoảng tin cậy
fig_cv, axes_cv = plt.subplots(1, 2, figsize=(14, 5))
fig_cv.suptitle('Ket qua Cross-Validation (5-Fold)', fontsize=14, fontweight='bold')

# Chạy lại để lấy từng fold score
cv_fold_scores = {}
for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv,
                            scoring='f1_macro', n_jobs=-1)
    cv_fold_scores[name] = scores['test_score']

# Left: Bar chart với error bar
ax_cv1 = axes_cv[0]
means = [cv_fold_scores[m].mean() for m in mnames]
stds  = [cv_fold_scores[m].std()  for m in mnames]
bc_cv = ['#f59e0b' if m == best_name else '#3b82f6' for m in mnames]
bars_cv = ax_cv1.bar(SHORT, means, color=bc_cv, edgecolor='white',
                      yerr=stds, capsize=6, error_kw={'linewidth':2})
ax_cv1.set_title('CV F1-Macro trung binh (±std)', fontweight='bold')
ax_cv1.set_ylabel('F1-Macro')
ax_cv1.set_ylim(max(0, min(means)-0.1), 1.0)
for bar, m, s in zip(bars_cv, means, stds):
    ax_cv1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+s+0.005,
                f'{m:.3f}', ha='center', fontsize=10, fontweight='bold')

# Right: Box plot phân phối 5 fold
ax_cv2 = axes_cv[1]
data_box = [cv_fold_scores[m] for m in mnames]
bp = ax_cv2.boxplot(data_box, labels=SHORT, patch_artist=True, notch=False)
for patch, color in zip(bp['boxes'], bc_cv):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax_cv2.set_title('Phan phoi F1-Macro qua 5 Fold', fontweight='bold')
ax_cv2.set_ylabel('F1-Macro')
ax_cv2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('cross_validation.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Da luu: cross_validation.png")

---
## ☁️ Bước 5c: WordCloud — Từ Đặc Trưng Theo Cảm Xúc

Từ càng **lớn** = xuất hiện càng nhiều trong nhóm cảm xúc đó.

In [ ]:
# [5c] WordCloud
from wordcloud import WordCloud

CMAP   = {0: 'Reds', 1: 'Greys', 2: 'Greens'}
LBL_WC = {0: 'Negative 😞', 1: 'Neutral 😐', 2: 'Positive 😊'}

unique_labels = sorted(df_train['sentiment'].unique())
fig_wc, axes  = plt.subplots(1, len(unique_labels),
                              figsize=(7*len(unique_labels), 5))
if len(unique_labels) == 1:
    axes = [axes]

for ax, label in zip(axes, unique_labels):
    corpus = ' '.join(df_train[df_train['sentiment']==label]['processed'].dropna())
    if not corpus.strip():
        ax.axis('off'); continue
    wc = WordCloud(width=700, height=450, background_color='white',
                   colormap=CMAP.get(label, 'Blues'),
                   max_words=100, collocations=False).generate(corpus)
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(LBL_WC.get(label, str(label)), fontsize=15, fontweight='bold')
    ax.axis('off')

fig_wc.suptitle('WordCloud — Tu Pho Bien Theo Cam Xuc Phan Hoi Sinh Vien',
                fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('wordcloud_sentiment.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Da luu: wordcloud_sentiment.png")

---
## 🔬 Bước 6: Phân Tích Lỗi (Error Analysis)

**Tại sao mô hình dự đoán sai?** Đây là bước quan trọng để hiểu giới hạn của mô hình và đề xuất cải tiến.

Phân tích lỗi giúp trả lời:
- Mô hình hay nhầm lẫn giữa các lớp nào nhất?
- Đặc điểm của các câu bị dự đoán sai là gì?
- Neutral có phải lớp khó nhất không?

In [ ]:
# [6.1] Phân tích tổng quan lỗi
df_test_analysis = df_test.copy()
df_test_analysis['true_label']      = y_test.values
df_test_analysis['pred_label']      = y_pred_best
df_test_analysis['true_name']       = df_test_analysis['true_label'].map(LABEL)
df_test_analysis['pred_name']       = df_test_analysis['pred_label'].map(LABEL)
df_test_analysis['is_correct']      = df_test_analysis['true_label'] == df_test_analysis['pred_label']

n_total   = len(df_test_analysis)
n_correct = df_test_analysis['is_correct'].sum()
n_wrong   = n_total - n_correct

print("=" * 60)
print("🔬 PHAN TICH LOI (ERROR ANALYSIS)")
print("=" * 60)
print(f"  Tong so mau test    : {n_total:,}")
print(f"  Du doan dung        : {n_correct:,} ({n_correct/n_total:.1%})")
print(f"  Du doan sai         : {n_wrong:,}  ({n_wrong/n_total:.1%})")
print()

# Ma trận nhầm lẫn chi tiết
print("  Phan tich nham lan giua cac cap nhan:")
print(f"  {'Thuc te → Du doan':<30} {'So luong':>9} {'Ti le sai':>10}")
print("  " + "-" * 52)
errors = df_test_analysis[~df_test_analysis['is_correct']]
error_pairs = errors.groupby(['true_name', 'pred_name']).size().reset_index(name='count')
error_pairs = error_pairs.sort_values('count', ascending=False)

for _, row in error_pairs.iterrows():
    total_true = (df_test_analysis['true_name'] == row['true_name']).sum()
    pct = row['count'] / total_true * 100
    print(f"  {row['true_name']} → {row['pred_name']:<20} {row['count']:>9} ({pct:>6.1f}% cua lop {row['true_name']})")

print()
print("  💡 Nhan xet:")
top_error = error_pairs.iloc[0]
print(f"  → Nham lan pho bien nhat: {top_error['true_name']} bi du doan thanh {top_error['pred_name']} ({top_error['count']} truong hop)")
print(f"  → Neutral thuong la lop kho phan loai nhat vi cam xuc trung lap,")
print(f"    khong co tu dac trung ro rang nhu Positive hay Negative.")

In [ ]:
# [6.2] Xem ví dụ các câu bị dự đoán sai theo từng loại lỗi
print("📋 VI DU CAC CAU DU DOAN SAI:")
print("=" * 70)

error_pairs_list = error_pairs.values.tolist()
for row in error_pairs_list[:4]:  # Xem 4 loại lỗi phổ biến nhất
    true_n, pred_n, count = row
    subset = df_test_analysis[
        (df_test_analysis['true_name'] == true_n) &
        (df_test_analysis['pred_name'] == pred_n)
    ]['sentence'].head(3).tolist()

    print(f"\n  ❌ Thuc te: {true_n} | Du doan sai thanh: {pred_n} ({count} truong hop)")
    print("  " + "-" * 60)
    for i, s in enumerate(subset, 1):
        print(f"  [{i}] {s[:100]}")

In [ ]:
# [6.3] Phân tích đặc điểm câu sai: độ dài và từ phủ định
print("📊 DAC DIEM CAU DU DOAN SAI vs DUNG:")
print("=" * 55)

# Độ dài câu
df_test_analysis['sent_len'] = df_test_analysis['sentence'].apply(
    lambda x: len(str(x).split()))

correct_len = df_test_analysis[df_test_analysis['is_correct']]['sent_len'].mean()
wrong_len   = df_test_analysis[~df_test_analysis['is_correct']]['sent_len'].mean()
print(f"  Do dai trung binh cau DUNG  : {correct_len:.1f} tu")
print(f"  Do dai trung binh cau SAI   : {wrong_len:.1f} tu")
print(f"  → Cau ngan hon thuong kho phan loai hon do it tu dac trung")

# Tỷ lệ sai theo từng lớp thực tế
print()
print("  Ti le du doan sai theo tung lop thuc te:")
for lbl in sorted(LABEL.keys()):
    lbl_name = LABEL[lbl]
    subset = df_test_analysis[df_test_analysis['true_label'] == lbl]
    err_rate = 1 - subset['is_correct'].mean()
    bar = '█' * int(err_rate * 30)
    print(f"  {lbl_name:<10}: {err_rate:>6.1%} {bar}")

print()
print("  💡 Goi y cai thien:")
print("  → Neutral kho vi thieu tu dac trung — co the thu them features")
print("    nhu so luong tu cam xuc (lexicon-based features)")
print("  → Xem xet dung PhoBERT de hieu ngu canh sau hon")

---
## ✅ Bước 7: Đánh Giá Trên Tập Validation

Tập **Validation** dùng để kiểm tra xem mô hình có bị **overfit** trên tập train không.  
So sánh: Train score ≈ Val score ≈ Test score → mô hình tổng quát hóa tốt.

In [ ]:
# [7.1] Đánh giá trên tập Validation (nếu có)
if df_val is not None and X_val is not None:
    print("=" * 60)
    print("📊 DANH GIA TREN TAP VALIDATION")
    print("=" * 60)
    print(f"{'Mo hinh':<22} {'Train F1':>10} {'Val F1':>10} {'Test F1':>10}")
    print("-" * 60)

    for name, model in models.items():
        # Train score
        yp_train = model.predict(X_train)
        f1_train = f1_score(y_train, yp_train, average='macro')
        # Val score
        yp_val   = model.predict(X_val)
        f1_val   = f1_score(y_val, yp_val, average='macro')
        # Test score
        f1_test  = results[name]['f1_macro']

        gap = abs(f1_train - f1_test)
        note = " ⚠️ overfit?" if gap > 0.05 else " ✅"
        print(f"  {name:<20} {f1_train:>10.4f} {f1_val:>10.4f} {f1_test:>10.4f}{note}")

    print("-" * 60)
    print("  Chenh lech Train-Test < 0.05: mo hinh tong quat hoa tot")
    print("  Chenh lech Train-Test > 0.05: co the bi overfit")
else:
    print("⚠️  Khong co tap Validation trong dataset nay.")
    print("   Ket qua Cross-Validation o Buoc 4b da thay the chuc nang nay.")

---
## 🔍 Bước 8: Demo Dự Đoán Câu Mới

In [ ]:
# [8.1] Hàm dự đoán cảm xúc
def predict_sentiment(text, model_name=None):
    """Dự đoán cảm xúc cho câu phản hồi mới."""
    name    = model_name or best_name
    model   = results[name]['model']
    vec     = tfidf.transform([preprocess_text(text)])
    pred    = model.predict(vec)[0]
    emap    = {0:'😞 Negative (Tieu cuc)', 1:'😐 Neutral (Trung lap)', 2:'😊 Positive (Tich cuc)'}
    return emap.get(pred, str(pred))

test_cases = [
    "Thay giang bai rat hay, de hieu va nhiet tinh ho tro sinh vien",
    "Mon hoc binh thuong, khong co gi dac biet",
    "Bai giang qua kho hieu, toi khong nam duoc kien thuc gi",
    "Giao vien giai thich ro rang tung buoc, toi rat thich mon nay",
    "Can cai thien phuong phap giang day, noi dung con nham chan",
    "Khong hieu tai sao phai hoc mon nay, hoan toan vo ich",
    "Thay co tan tam, luon ho tro sinh vien kip thoi"
]

print(f"🧪 Demo du doan — Mo hinh: {best_name}")
print("=" * 72)
for i, s in enumerate(test_cases, 1):
    print(f"  [{i}] {s}")
    print(f"       → {predict_sentiment(s)}")
    print()

---
## 💡 Bước 9: Phân Tích Insight & Báo Cáo Cải Thiện Giảng Dạy

In [ ]:
# [9.1] Báo cáo tổng hợp
df_test['predicted_sentiment'] = y_pred_best
df_test['predicted_label']     = df_test['predicted_sentiment'].map(LABEL)

total     = len(df_test)
pos_count = sum(df_test['predicted_sentiment'] == 2)
neu_count = sum(df_test['predicted_sentiment'] == 1)
neg_count = sum(df_test['predicted_sentiment'] == 0)
pos_pct   = pos_count / total * 100
neu_pct   = neu_count / total * 100
neg_pct   = neg_count / total * 100

print("=" * 65)
print("📈  BAO CAO PHAN TICH PHAN HOI SINH VIEN")
print("=" * 65)
print(f"  Tong so phan hoi phan tich : {total:,}")
print(f"  Mo hinh su dung            : {best_name}")
print(f"  Accuracy (Test set)        : {results[best_name]['accuracy']:.4f}")
print(f"  F1-Macro (Test set)        : {results[best_name]['f1_macro']:.4f}")
print(f"  CV F1-Macro (5-Fold)       : {cv_results[best_name]['f1_mean']:.4f} ± {cv_results[best_name]['f1_std']:.4f}")
print()
print("  Phan bo cam xuc du doan:")
for name_l, count, pct, ch in [
    ('😊 Positive (Tich cuc)', pos_count, pos_pct, '█'),
    ('😐 Neutral  (Trung lap)', neu_count, neu_pct, '▒'),
    ('😞 Negative (Tieu cuc)', neg_count, neg_pct, '░'),
]:
    bar = ch * int(pct / 2)
    print(f"  {name_l}: {count:4d} ({pct:5.1f}%) {bar}")

print()
print("-" * 65)
print("💡 GOI Y CAI THIEN CHAT LUONG GIANG DAY:")
print("-" * 65)

if neg_pct > 35:
    print("  ⚠️  Ti le phan hoi TIEU CUC CAO (>35%). Can xem xet:")
    print("     • Ra soat phuong phap va toc do giang day")
    print("     • Tang cuong tuong tac, hoi dap trong gio hoc")
    print("     • Don gian hoa noi dung hoac bo sung tai lieu tham khao")
    print("     • To chuc buoi phan hoi truc tiep voi sinh vien")
elif pos_pct >= 60:
    print("  ✅ Phan hoi TICH CUC chiem da so (>=60%). Tiep tuc phat huy:")
    print("     • Duy tri phong cach giang day hien tai")
    print("     • Bo sung them bai tap thuc hanh de tang ky nang")
    print("     • Chia se kinh nghiem giang day voi dong nghiep")
else:
    print("  📊 Phan hoi o muc TRUNG BINH. Co the cai thien:")
    print("     • Da dang hoa phuong phap giang day")
    print("     • Thu thap them y kien chi tiet tu sinh vien")
    print("     • Them vi du thuc te gan voi chuyen nganh")

print()
print(f"  ➡️  Ti le hai long (Positive)      : {pos_pct:.1f}%")
print(f"  ➡️  Ti le chua hai long (Negative) : {neg_pct:.1f}%")
print("=" * 65)

---
## ✅ Tổng Kết

### Kết quả đạt được:

| Bước | Nội dung | Chi tiết |
|------|----------|----------|
| 0 | Môi trường | datasets==2.18.0, scikit-learn, matplotlib, wordcloud |
| 1 | Dataset | UIT-VSFC — ~16,000 phản hồi sinh viên, 3 nhãn cảm xúc |
| 2 | Tiền xử lý | Lowercase, khử nhiễu, stopwords (giữ từ phủ định) |
| 3 | TF-IDF | 15,000 features, unigram+bigram, sublinear_tf, max_df |
| 4 | Mô hình | So sánh LR / MNB / CNB / SVM — Accuracy & F1-Macro |
| 4b | Cross-Validation | 5-Fold Stratified CV — kiểm chứng độ ổn định |
| 4c | Ablation Study | So sánh unigram / bigram / trigram |
| 5 | Trực quan | 9 biểu đồ + CV chart + WordCloud |
| 6 | Error Analysis | Phân tích câu sai, nhầm lẫn giữa các lớp |
| 7 | Validation | So sánh Train / Val / Test để phát hiện overfit |
| 8 | Demo | Dự đoán real-time câu phản hồi bất kỳ |
| 9 | Báo cáo | Thống kê + gợi ý cải thiện chất lượng giảng dạy |

### Hướng phát triển tiếp theo:
- 🤗 Dùng **PhoBERT** (BERT tiếng Việt của VinAI) để tăng accuracy đáng kể
- 📝 Tích hợp **Underthesea** để tách từ tiếng Việt chính xác hơn
- 📊 Xây dựng **dashboard** (Streamlit/Gradio) để triển khai thực tế
- 📚 Bổ sung **sentiment lexicon** tiếng Việt làm feature thêm

---
> *Hệ thống sử dụng các kỹ thuật NLP cổ điển: TF-IDF kết hợp Machine Learning.  
> Kết quả phân tích hỗ trợ giảng viên và nhà trường đưa ra quyết định cải thiện  
> chất lượng giảng dạy dựa trên dữ liệu thực tế từ phản hồi sinh viên.*